<font face="Comic Sans MS">Импорт необходимых библиотек:</font>

In [ ]:
import multiprocessing

import albumentations as Alb
import kagglehub
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from pytorch_metric_learning.losses import ArcFaceLoss
from torch.amp import GradScaler, autocast
from torch.optim import AdamW, lr_scheduler
from torch.utils.data import DataLoader
from tqdm import tqdm

Загрузка датасета [Veri](https://www.kaggle.com/datasets/abhyudaya12/veri-vehicle-re-identification-dataset)

In [ ]:
path_veri_dataset = kagglehub.dataset_download("abhyudaya12/veri-vehicle-re-identification-dataset")

Создание датасета (ВНИМАНИЕ! технически возможно только в отдельном файле)

import os
from torch.utils.data import Dataset
from PIL import Image
import numpy as np


class VeRiDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Аргументы:
            root_dir (string): Путь к папке со всеми картинками.\n
            transform (callable, optional): Аугментации (ресайз, нормализация).
        """
        super().__init__()

        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [file for file in os.listdir(root_dir)]
        self.raw_labels = []

        for filename in self.image_files:
            car_id = filename.split('_')[0]
            self.raw_labels.append(car_id)

        unique_ids = sorted(list(set(self.raw_labels)))
        self.id_to_index = {raw_id: i for i, raw_id in enumerate(unique_ids)}
        self.num_classes = len(unique_ids)
        print(f'Всего изображений: {len(self.image_files)}')
        print(f'Уникальных машин (классов): {self.num_classes}')

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        img_name = self.image_files[index]
        img_path = f'{self.root_dir}/{img_name}'

        image = Image.open(img_path).convert('RGB')
        image_np = np.array(image)

        raw_id = self.raw_labels[index]
        label = self.id_to_index[raw_id]

        if self.transform:
            augment = self.transform(image=image_np)
            image = augment['image']

        return image, label

In [ ]:
from VeriDataset import VeRiDataset

Функция аугментации для ReID

In [ ]:
train_transforms = Alb.Compose([
    Alb.LongestMaxSize(max_size=600),
    Alb.PadIfNeeded(min_height=600, min_width=600, border_mode=cv2.BORDER_CONSTANT),
    Alb.HorizontalFlip(p=0.5),
    Alb.CoarseDropout(
        num_holes_range=(1, 8),
        hole_height_range=(10, 64),
        hole_width_range=(10, 64),
        fill=255,
        p=0.5
    ),

    Alb.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

Создание загрузчика

In [ ]:
veri_dataset = VeRiDataset(root_dir=r'C:\Users\Homo\.cache\kagglehub\datasets\abhyudaya12\veri-vehicle-re-identification-dataset\versions\1\VeRi\image_train', transform=train_transforms)

multiprocessing.set_start_method("spawn", force=True)

train_loader_veri = DataLoader(
    dataset=veri_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=12,
    pin_memory=True,
    persistent_workers=True,
)

Функция обучения

In [ ]:
def train_car_detect(model, train_loader, epochs=15, lr=3e-4):

    device = 'cuda:0'
    model.to(device)

    loss_func = ArcFaceLoss(
        num_classes=train_loader.dataset.num_classes, 
        embedding_size=512, 
        margin=28.6,
        scale=64
    ).to(device)

    optimizer = AdamW([
        {'params': model.parameters()},
        {'params': loss_func.parameters()}
    ], lr=lr, weight_decay=1e-4)

    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)

    scaler = GradScaler(device)

    pbar = tqdm(range(epochs))
    best_loss = float('inf')

    model.train()
    for epoch in pbar:

        losses = []

        for images, labels in enumerate(train_loader):

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            with autocast(device, dtype=torch.float16):
                embeddings = model(images)
                loss = loss_func(embeddings, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            losses.append(loss.item())

        scheduler.step()

        epoch_loss = torch.Tensor(losses).mean().item()

        if epoch_loss < best_loss:
            best_loss = epoch_loss

            torch.save(model.state_dict(), 'best_car_veri_backbone.pth')
            torch.save(loss_func.state_dict(), 'best_car_veri_centers.pth')

        pbar.set_postfix({'loss': epoch_loss, 'best_loss': best_loss})

    pbar.close()
    torch.save(model.state_dict(), 'last_car_veri_backbone.pth')
    torch.save(loss_func.state_dict(), 'last_car_veri_centers.pth')

Загрузка модели [ResNet50-IBN](https://github.com/XingangPan/IBN-Net). Возможно в будущем стоит рассмотреть вариант [ResNet50-vd](https://huggingface.co/keras/resnet_vd_50_imagenet)

In [ ]:
def get_resnet50_ibn_modification_arcface():
    model = torch.hub.load('XingangPan/IBN-Net', 'resnet50_ibn_a', pretrained=True, verbose=False)
    model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    model.fc = nn.Sequential(
    nn.Linear(2048, 512, bias=False),
    nn.BatchNorm1d(512)
)
    return model

Обучение

In [ ]:
train_car_detect(get_resnet50_ibn_modification_arcface(), train_loader_veri, epochs=60, lr=3e-4)